<a href="https://colab.research.google.com/github/DunyaVatandasi/abd-borsa-calismalari/blob/main/02_haber_duyarlilik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance transformers

import yfinance as yf
import pandas as pd
from transformers import pipeline

# FinBERT modelini yükle (ilk seferde 1-2 dakika sürebilir)
finbert = pipeline("text-classification", model="ProsusAI/finbert")

def haber_duyarlilik(sembol, adet=10):
    haberler = yf.Ticker(sembol).news[:adet]
    satirlar = []
    for h in haberler:
        icerik = h.get("content", h)
        baslik = icerik.get("title", "")
        if not baslik:
            continue
        sonuc = finbert(baslik)[0]
        satirlar.append({
            "Tarih": icerik.get("pubDate", "")[:10],
            "Kaynak": icerik.get("provider", {}).get("displayName", ""),
            "Başlık": baslik,
            "Duyarlılık": sonuc["label"],
            "Güven": round(sonuc["score"], 2),
        })
    tablo = pd.DataFrame(satirlar)
    print(f"===== {sembol} haber duyarlılığı =====")
    print(tablo["Duyarlılık"].value_counts(), "\n")
    return tablo

pd.set_option("display.max_colwidth", 100)
haber_duyarlilik("AAPL")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

RuntimeError: *** YAHOO! FINANCE IS CURRENTLY DOWN! ***
Our engineers are working quickly to resolve the issue. Thank you for your patience.

In [2]:
!pip install -q feedparser
import feedparser, urllib.parse

def haberleri_getir(sembol, adet=10):
    # 1) Önce Yahoo Finance'i dene
    try:
        sonuc = []
        for h in yf.Ticker(sembol).news[:adet]:
            icerik = h.get("content", h)
            if icerik.get("title"):
                sonuc.append({
                    "Tarih": icerik.get("pubDate", "")[:10],
                    "Kaynak": icerik.get("provider", {}).get("displayName", ""),
                    "Başlık": icerik["title"],
                })
        if sonuc:
            return sonuc, "Yahoo Finance"
    except Exception:
        pass

    # 2) Yahoo engellerse yedek: Google News
    sorgu = urllib.parse.quote(f"{sembol} stock")
    url = f"https://news.google.com/rss/search?q={sorgu}&hl=en-US&gl=US&ceid=US:en"
    sonuc = []
    for e in feedparser.parse(url).entries[:adet]:
        sonuc.append({
            "Tarih": pd.to_datetime(e.published).strftime("%Y-%m-%d"),
            "Kaynak": e.get("source", {}).get("title", ""),
            "Başlık": e.title.rsplit(" - ", 1)[0],  # sondaki yayıncı adını at
        })
    return sonuc, "Google News"

def haber_duyarlilik(sembol, adet=10):
    haberler, kaynak = haberleri_getir(sembol, adet)
    for h in haberler:
        s = finbert(h["Başlık"])[0]
        h["Duyarlılık"] = s["label"]
        h["Güven"] = round(s["score"], 2)
    tablo = pd.DataFrame(haberler)
    print(f"===== {sembol} haber duyarlılığı (kaynak: {kaynak}) =====")
    print(tablo["Duyarlılık"].value_counts(), "\n")
    return tablo

haber_duyarlilik("AAPL")


===== AAPL haber duyarlılığı (kaynak: Google News) =====
Duyarlılık
neutral     8
negative    2
Name: count, dtype: int64 



,Tarih,Kaynak,Başlık,Duyarlılık,Güven
0,2026-11-14,Yahoo Finance,Is Apple Stock a Buy Right Now?,neutral,0.91
1,2026-11-11,CNN,AAPL Stock Quote Price and Forecast,neutral,0.94
2,2026-11-14,The Motley Fool,Is Apple Stock a Buy Right Now?,neutral,0.91
3,2026-11-11,Barron's,Why the iPhone Duo Could Kick Off a ‘New Decade’ for Apple,neutral,0.66
4,2026-11-13,Yahoo Finance,What Could Still Send Apple Stock Higher?,negative,0.70
5,2026-11-11,Benzinga,What's Going On With Apple Stock Wednesday?,negative,0.55
6,2026-11-13,The Motley Fool,"Apple Is the World’s Largest Company, and Nvidia Is Second. Here’s What $10,000 Invested in Each...",neutral,0.94
7,2026-11-11,Yahoo Finance,Apple Shares (AAPL) Look Expensive Heading into 2027,neutral,0.75
8,2026-11-09,The Motley Fool,Where Will Apple Stock Be in 5 Years?,neutral,0.93
9,2026-11-09,Yahoo Finance,Where Will Apple Stock Be in 5 Years?,neutral,0.93


In [3]:
import requests

def stocktwits_duyarlilik(sembol):
    url = f"https://api.stocktwits.com/api/2/streams/symbol/{sembol}.json"
    basliklar = {"User-Agent": "Mozilla/5.0"}  # kendimizi tarayıcı gibi tanıtıyoruz

    try:
        yanit = requests.get(url, headers=basliklar, timeout=15)
    except Exception as e:
        print(f"Bağlantı kurulamadı: {e}")
        return None

    if yanit.status_code != 200:
        print(f"StockTwits erişimi reddetti (kod: {yanit.status_code}).")
        return None

    satirlar = []
    for m in yanit.json().get("messages", []):
        etiket = (m.get("entities", {}).get("sentiment") or {}).get("basic")
        satirlar.append({
            "Tarih": m.get("created_at", "")[:16].replace("T", " "),
            "Kullanıcı": m.get("user", {}).get("username", ""),
            "Takipçi": m.get("user", {}).get("followers", 0),
            "Etiket": etiket or "Etiketsiz",
            "Mesaj": m.get("body", ""),
        })

    tablo = pd.DataFrame(satirlar)
    sayim = tablo["Etiket"].value_counts()
    boga = sayim.get("Bullish", 0)
    ayi = sayim.get("Bearish", 0)

    print(f"===== {sembol} StockTwits duyarlılığı (son {len(tablo)} mesaj) =====")
    print(sayim, "\n")
    if boga + ayi > 0:
        print(f"Boğa oranı (etiketli mesajlarda): %{boga / (boga + ayi) * 100:.0f}")
    return tablo

stocktwits_duyarlilik("AAPL")

===== AAPL StockTwits duyarlılığı (son 30 mesaj) =====
Etiket
Etiketsiz    21
Bearish       5
Bullish       4
Name: count, dtype: int64 

Boğa oranı (etiketli mesajlarda): %44


,Tarih,Kullanıcı,Takipçi,Etiket,Mesaj
0,2026-11-14 06:59,Diehard777,7,Etiketsiz,$AAPL $SPY believe Apple phones may be collecting personal information from people all over th...
1,2026-11-14 06:29,AcceleratorModel,238,Etiketsiz,$AAPL $NVDA \n\nNVDA — Should snap back to that .2 - .4 positive corr\nSoon\n\nPast 2 weeks ther...
2,2026-11-14 03:20,beestMod516,482,Etiketsiz,Soros portfolio 🤔 \n$AAPL $NVDA $AMZN $GOOG $SPY
3,2026-11-14 02:56,PivotPoint_101Official,5335,Etiketsiz,"$SPY $AAPL $BTC.X\n\nOur traders had a splendid week, \nyour traders are filing for divorce."
4,2026-11-14 02:52,PivotPoint_101Official,5335,Etiketsiz,$SPY $QQQ $AAPL $NVDA \n#pivotgroup💸\n#woofstreets🎬
5,2026-11-14 02:48,SamsonCapital,48250,Etiketsiz,$AMZN $AAPL $NVDA $GOOGL $TKO \n\nSoros Fund Management LLC top holdings:
6,2026-11-14 02:45,LeonardoBonacci,998,Bearish,$QQQ Fib 61.8% rejection played well today. It might not feel like it but this week&#39;s weekl...
7,2026-11-14 02:09,OptionsPlayers,17789,Etiketsiz,$AAPL $QQQ We put in work\nhttps://www.instagram.com/stories/optionsplayers/4029602271980981099?...
8,2026-11-14 02:03,BullseyeGains,12317,Etiketsiz,$AAPL $GOOGL $GOOG $MSFT $AMZN Apple is up 30% this year and now trades at about 38 times forwar...
9,2026-11-14 01:41,earnit_saveit_investit,36,Etiketsiz,$NVDA $TGT $SPOT $AAPL $KR \n \nRECENT CEOs WHO LEFT THEIR COMPANY \nWHERE WILL NVIDIA END UP? ...
